# NEC One-Configuration Multi-Dataset Benchmark

Muc tieu:
- Dung mot configuration duy nhat (`cfg_one`) cho tat ca dataset.
- Giu nguyen kien truc 2 networks (CalibNet + EmuNet).
- Danh gia MAE/RMSE va metric chuan hoa tren 6 khu vuc.


In [ ]:
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.autograd import grad

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
def parse_pair_matrix(df):
    arr = df.values.astype(np.float32)
    if arr.shape[0] != 2:
        raise ValueError(f'Expect 2 rows (time, value), got shape={arr.shape}')
    t = arr[0]
    y = arr[1]
    return t, y


def load_split(train_path, val_path):
    tr = pd.read_csv(train_path, header=None)
    va = pd.read_csv(val_path, header=None)
    t_train, y_train = parse_pair_matrix(tr)
    t_val, y_val = parse_pair_matrix(va)

    idx_tr = np.argsort(t_train)
    idx_va = np.argsort(t_val)
    t_train, y_train = t_train[idx_tr], y_train[idx_tr]
    t_val, y_val = t_val[idx_va], y_val[idx_va]

    return t_train.astype(np.float32), y_train.astype(np.float32), t_val.astype(np.float32), y_val.astype(np.float32)


def build_calib_features(y_train, dim=10):
    y_log = np.log1p(y_train)
    idx = np.linspace(0, len(y_log) - 1, dim).astype(int)
    feat = y_log[idx]
    feat = (feat - feat.mean()) / (feat.std() + 1e-6)
    return feat.astype(np.float32)


In [ ]:
param_bounds = {
    'lambda': (0.0, 0.6),
    'betam': (0.0, 0.05),
    'mum': (0.001, 0.016),
    'thetam': (0.01, 0.2),
    'muh': (0.001, 0.0195),
    'betah': (0.001, 0.055),
    'thetah': (0.01, 0.53),
    'gammah': (0.05, 0.1),
}

CALIB_INPUT_DIM = 10


class CalibNet(nn.Module):
    def __init__(self, bounds):
        super().__init__()
        self.fc1 = nn.Linear(CALIB_INPUT_DIM, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, len(bounds))
        self.bounds = bounds
        self.names = list(bounds.keys())

    def forward(self, x):
        h = F.gelu(self.fc1(x))
        h = F.gelu(self.fc2(h))
        raw = self.fc3(h)
        params = {}
        for i, name in enumerate(self.names):
            lo, hi = self.bounds[name]
            params[name] = torch.tanh(raw[..., i]) * (hi - lo) / 2 + (hi + lo) / 2
        return params


class EmuNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(1, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 32),
        ])
        self.out = nn.Linear(32, 8)

    def forward(self, t_batch):
        h = t_batch
        for layer in self.layers:
            h = F.gelu(layer(h))
        return F.softplus(self.out(h))


class NECDual(nn.Module):
    def __init__(self, t_train, y_train, calib_features=None):
        super().__init__()
        self.device = device
        self.t = torch.tensor(t_train, dtype=torch.float32, requires_grad=True, device=self.device)
        self.t_batch = self.t.reshape(-1, 1)
        self.y = torch.tensor(y_train, dtype=torch.float32, device=self.device)
        self.y_log = torch.log1p(self.y)
        self.t_max = float(t_train[-1])

        self.calib = CalibNet(param_bounds).to(self.device)
        self.emu = EmuNet().to(self.device)

        if calib_features is None:
            self.calib_input = torch.zeros(1, CALIB_INPUT_DIM, dtype=torch.float32, device=self.device)
        else:
            self.calib_input = torch.tensor(calib_features, dtype=torch.float32, device=self.device).reshape(1, -1)

    def ode_residuals(self, params, t, t_batch):
        lam = params['lambda']
        betam = params['betam']
        mum = params['mum']
        thetam = params['thetam']
        muh = params['muh']
        betah = params['betah']
        thetah = params['thetah']
        gammah = params['gammah']

        y_hat = self.emu(t_batch)
        Hr_hat = y_hat[:, 0:1]
        Hs_hat = y_hat[:, 1:2]
        He_hat = y_hat[:, 2:3]
        Hi_hat = y_hat[:, 3:4]
        Me_hat = y_hat[:, 4:5]
        Ms_hat = y_hat[:, 5:6]
        Mi_hat = y_hat[:, 6:7]
        Itot_hat = y_hat[:, 7:8]

        Hr_t = grad(Hr_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Hs_t = grad(Hs_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        He_t = grad(He_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Hi_t = grad(Hi_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Me_t = grad(Me_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Ms_t = grad(Ms_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Mi_t = grad(Mi_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Itot_t = grad(Itot_hat.sum(), t, retain_graph=True, create_graph=True)[0]

        Hr = torch.expm1(Hr_hat.squeeze(1))
        Hs = torch.expm1(Hs_hat.squeeze(1))
        He = torch.expm1(He_hat.squeeze(1))
        Hi = torch.expm1(Hi_hat.squeeze(1))
        Me = torch.expm1(Me_hat.squeeze(1))
        Ms = torch.expm1(Ms_hat.squeeze(1))
        Mi = torch.expm1(Mi_hat.squeeze(1))
        Itot = torch.expm1(Itot_hat.squeeze(1))

        H = torch.clamp(Hr + Hs + He + Hi, min=1e-8)
        M = torch.clamp(Me + Ms + Mi, min=1e-8)

        f1 = Hr_t - (gammah * Hi - muh * Hr) / (1.0 + Hr)
        # f2 sign follows paper Model (3)
        f2 = Hs_t - (muh * H - betah * Mi * Hs / M - muh * Hs) / (1.0 + Hs)
        f3 = He_t - (betah * Mi * Hs / M - thetah * He - muh * He) / (1.0 + He)
        f4 = Hi_t - (thetah * He - gammah * Hi - muh * Hi) / (1.0 + Hi)
        f5 = Me_t - (betam * Hi * Ms / H - thetam * Me - mum * Me) / (1.0 + Me)
        f6 = Ms_t - (lam - betam * Hi * Ms / H - mum * Ms) / (1.0 + Ms)
        f7 = Mi_t - (thetam * Me - mum * Mi) / (1.0 + Mi)
        f8 = Itot_t - (thetah * He) / (1.0 + Itot)

        return f1, f2, f3, f4, f5, f6, f7, f8, Itot_hat.squeeze(1)

    def netf_data(self):
        params = self.calib(self.calib_input)
        return self.ode_residuals(params, self.t, self.t_batch)

    def netf_colloc(self, n_colloc=160):
        t_col = torch.rand(n_colloc, device=self.device) * self.t_max
        t_col.requires_grad_(True)
        t_col_batch = t_col.reshape(-1, 1)
        params = self.calib(self.calib_input)
        out = self.ode_residuals(params, t_col, t_col_batch)
        return out[:8]

    def predict_itot_numpy(self, t_np):
        t_tensor = torch.tensor(t_np, dtype=torch.float32, device=self.device).reshape(-1, 1)
        with torch.no_grad():
            y_log = self.emu(t_tensor)[:, 7]
            y = torch.expm1(y_log)
        return y.detach().cpu().numpy()


In [ ]:
def rel_rmse(y_true, y_pred):
    den = np.sum(np.square(y_true))
    if den <= 1e-12:
        return np.nan
    return float(np.sqrt(np.sum((y_true - y_pred) ** 2) / den))


def quantile_cap(values, q=0.9, mult=1.2, floor=1e-6):
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return float(floor)
    return float(max(np.quantile(arr, q) * mult, floor))


def estimate_tail_caps(y_train, cfg):
    dy = np.diff(y_train)
    if len(dy) == 0:
        return 1.0, 1e-3

    growth = dy / (y_train[:-1] + 1.0)
    tail_w = int(cfg.get('cap_tail_window', 20))
    tail_w = max(4, min(tail_w, len(dy)))

    dy_tail = dy[-tail_w:]
    g_tail = growth[-tail_w:]

    dy_cap_tail = quantile_cap(dy_tail[dy_tail > 0.0], q=float(cfg.get('spike_cap_q_tail', 0.95)), mult=float(cfg.get('spike_cap_mult_tail', 1.8)), floor=1.0)
    g_cap_tail = quantile_cap(g_tail[g_tail > 0.0], q=float(cfg.get('growth_cap_q_tail', 0.95)), mult=float(cfg.get('growth_cap_mult_tail', 2.0)), floor=1e-4)

    dy_cap_global = quantile_cap(dy[dy > 0.0], q=float(cfg.get('spike_cap_q_global', 0.80)), mult=float(cfg.get('spike_cap_mult_global', 0.70)), floor=1.0)
    g_cap_global = quantile_cap(growth[growth > 0.0], q=float(cfg.get('growth_cap_q_global', 0.80)), mult=float(cfg.get('growth_cap_mult_global', 0.40)), floor=1e-4)

    dy_cap = max(dy_cap_tail, dy_cap_global)
    growth_cap = max(g_cap_tail, g_cap_global)
    return float(dy_cap), float(growth_cap)


def project_sequence(y_seq, y0, dy_cap, growth_cap, smooth_alpha=0.0):
    y = np.asarray(y_seq, dtype=np.float64)
    out = np.zeros_like(y)
    prev = max(float(y0), 0.0)
    alpha = float(np.clip(smooth_alpha, 0.0, 0.95))

    for i in range(len(y)):
        raw = max(float(y[i]), 0.0)
        max_step = min(float(dy_cap), float(growth_cap) * (prev + 1.0))
        max_step = max(max_step, 0.0)
        lo = prev
        hi = prev + max_step
        clipped = min(max(raw, lo), hi)
        if i > 0 and alpha > 0.0:
            clipped = alpha * out[i - 1] + (1.0 - alpha) * clipped
        out[i] = clipped
        prev = clipped

    return out.astype(np.float32)


def project_sequence_torch(y_seq, y0, dy_cap, growth_cap, smooth_alpha=0.0):
    y = y_seq.reshape(-1)
    device_local = y.device
    dtype_local = y.dtype

    prev = torch.clamp(torch.as_tensor(y0, device=device_local, dtype=dtype_local), min=0.0)
    dy_cap_t = torch.as_tensor(float(dy_cap), device=device_local, dtype=dtype_local)
    growth_cap_t = torch.as_tensor(float(growth_cap), device=device_local, dtype=dtype_local)
    alpha = float(np.clip(smooth_alpha, 0.0, 0.95))

    out = []
    for i in range(y.shape[0]):
        raw = torch.clamp(y[i], min=0.0)
        max_step = torch.minimum(dy_cap_t, growth_cap_t * (prev + 1.0))
        max_step = torch.clamp(max_step, min=0.0)
        clipped = torch.clamp(raw, min=prev, max=prev + max_step)
        if i > 0 and alpha > 0.0:
            clipped = alpha * out[-1] + (1.0 - alpha) * clipped
        out.append(clipped)
        prev = clipped

    return torch.stack(out)


def compute_phys_weights(loss_list, ema_state, cfg):
    vals = torch.stack([l.detach() for l in loss_list])
    if ema_state is None:
        ema_state = vals
    else:
        beta = float(cfg.get('phys_ema_beta', 0.95))
        ema_state = beta * ema_state + (1.0 - beta) * vals

    eps = float(cfg.get('phys_eps', 1e-8))
    inv = 1.0 / torch.clamp(ema_state, min=eps)
    inv = inv / torch.clamp(inv.mean(), min=eps)
    clip_max = float(cfg.get('phys_weight_clip', 10.0))
    inv = torch.clamp(inv, max=clip_max)
    temp = max(float(cfg.get('phys_temp', 1.0)), 1e-6)
    logits = inv / temp
    weights = torch.softmax(logits, dim=0) * len(loss_list)
    return weights.detach(), ema_state


def ramp_value(epoch, start_epoch, ramp_epochs, max_val):
    if epoch < start_epoch:
        return 0.0
    if epoch < start_epoch + max(1, ramp_epochs):
        return float(max_val) * (epoch - start_epoch) / float(max(1, ramp_epochs))
    return float(max_val)


In [ ]:
def run_experiment(region, cfg):
    seed_everything(SEED)
    train_path = f'data/{region}_train.csv'
    val_path = f'data/{region}_val.csv'
    t_train, y_train, t_val, y_val = load_split(train_path, val_path)

    features = build_calib_features(y_train, 10) if cfg['use_features'] else None
    model = NECDual(t_train, y_train, calib_features=features).to(device)

    params = list(model.calib.parameters()) + list(model.emu.parameters())
    optimizer = optim.Adam(params, lr=cfg['lr'], betas=(0.9, 0.999), weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=300, min_lr=1e-7)

    for p in model.calib.parameters():
        p.requires_grad = False

    dy_cap, growth_cap = estimate_tail_caps(y_train, cfg)
    dy_cap_t = torch.tensor(dy_cap, dtype=torch.float32, device=device)
    growth_cap_t = torch.tensor(growth_cap, dtype=torch.float32, device=device)

    eval_horizon = int(cfg.get('eval_horizon', 10))
    t_future_eval_np = np.arange(float(t_train[-1]) + 1.0, float(t_train[-1]) + 1.0 + eval_horizon, dtype=np.float32)
    t_future_eval_t = torch.tensor(t_future_eval_np, dtype=torch.float32, device=device).reshape(-1, 1)

    anchor_w = int(cfg.get('anchor_tail_window', 12))
    anchor_w = max(4, min(anchor_w, len(y_train) - 1))
    dy_anchor = np.diff(y_train[-(anchor_w + 1):])
    anchor_step = float(np.median(dy_anchor[-min(6, len(dy_anchor)) :])) if len(dy_anchor) > 0 else 0.0
    y_anchor0 = float(y_train[-1] + max(anchor_step, 0.0))
    y_anchor0_t = torch.tensor(y_anchor0, dtype=torch.float32, device=device)

    n_train = len(y_train)
    tw_start = float(cfg.get('time_weight_start', 0.30))
    time_w = torch.linspace(tw_start, 1.0, n_train, device=device)
    time_w = time_w / torch.clamp(time_w.mean(), min=1e-6)

    best_score = float('inf')
    best_state = None
    best_epoch = -1
    no_improve = 0
    ema_phys = None
    start = time.time()

    for epoch in range(cfg['epochs'] + 1):
        if epoch == cfg['warmup']:
            for p in model.calib.parameters():
                p.requires_grad = True

        if epoch < cfg['warmup']:
            lambda_phys = 0.0
        elif epoch < cfg['warmup'] + cfg['ramp']:
            lambda_phys = (epoch - cfg['warmup']) / max(1, cfg['ramp']) * cfg['lambda_phys_max']
        else:
            lambda_phys = cfg['lambda_phys_max']

        lambda_proj = ramp_value(epoch, cfg['proj_start_epoch'], cfg['proj_ramp_epochs'], cfg['lambda_proj'])
        lambda_roll = ramp_value(epoch, cfg['rollout_start_epoch'], cfg['rollout_ramp_epochs'], cfg['lambda_roll'])

        h_min = int(cfg.get('proj_horizon_min', 6))
        h_max = int(cfg.get('proj_horizon_max', eval_horizon))
        prog = max(0.0, min(1.0, (epoch - cfg['warmup']) / max(1, cfg['epochs'] - cfg['warmup'])))
        train_horizon = int(round(h_min + (h_max - h_min) * prog))
        train_horizon = max(4, train_horizon)

        t_future_train_np = np.arange(float(t_train[-1]) + 1.0, float(t_train[-1]) + 1.0 + train_horizon, dtype=np.float32)
        t_future_train_t = torch.tensor(t_future_train_np, dtype=torch.float32, device=device).reshape(-1, 1)

        optimizer.zero_grad()
        out_data = model.netf_data()

        err_log = out_data[8] - model.y_log
        beta = float(cfg.get('huber_beta', 0.45))
        ae = torch.abs(err_log)
        loss_log_vec = torch.where(ae <= beta, 0.5 * (ae ** 2) / beta, ae - 0.5 * beta)
        loss_data_log = torch.mean(time_w * loss_log_vec)

        y_pred_train = torch.expm1(out_data[8])
        rel_err = (y_pred_train - model.y) / torch.clamp(model.y + 500.0, min=1.0)
        loss_data_rel = torch.mean(time_w * (rel_err ** 2))

        dy_pred = y_pred_train[1:] - y_pred_train[:-1]
        dy_true = model.y[1:] - model.y[:-1]
        k_tail = int(cfg.get('trend_tail_k', 16))
        k_tail = max(4, min(k_tail, dy_pred.shape[0]))
        tail_scale = torch.clamp(torch.mean(torch.abs(dy_true[-k_tail:])), min=1.0)
        loss_trend = F.smooth_l1_loss(dy_pred[-k_tail:], dy_true[-k_tail:], beta=float(cfg.get('trend_beta', 1200.0))) / tail_scale

        loss_data = loss_data_log + cfg['lambda_data_rel'] * loss_data_rel + cfg['lambda_trend'] * loss_trend

        phys_data = [F.mse_loss(r, torch.zeros_like(r)) for r in out_data[:8]]
        out_col = model.netf_colloc(n_colloc=cfg['n_colloc'])
        phys_col = [F.mse_loss(r, torch.zeros_like(r)) for r in out_col]
        phys_terms = [0.5 * ld + 0.5 * lc for ld, lc in zip(phys_data, phys_col)]
        phys_weights, ema_phys = compute_phys_weights(phys_terms, ema_phys, cfg)
        loss_phys = torch.sum(phys_weights * torch.stack(phys_terms))

        y_future = torch.expm1(model.emu(t_future_train_t)[:, 7])
        dy_future = y_future[1:] - y_future[:-1]
        loss_mono = torch.mean(F.relu(-dy_future))

        loss_spike = torch.mean((F.relu(dy_future - dy_cap_t) / torch.clamp(dy_cap_t, min=1.0)) ** 2)
        growth_future = dy_future / torch.clamp(y_future[:-1] + 1.0, min=1.0)
        loss_growth = torch.mean((F.relu(growth_future - growth_cap_t) / torch.clamp(growth_cap_t, min=1e-4)) ** 2)

        if dy_future.shape[0] > 1:
            ddy_future = dy_future[1:] - dy_future[:-1]
            scale = torch.clamp(torch.mean(y_future), min=1.0)
            loss_smooth = torch.mean((ddy_future / scale) ** 2)
        else:
            loss_smooth = torch.tensor(0.0, device=device)

        loss_anchor = ((y_future[0] - y_anchor0_t) / torch.clamp(y_anchor0_t, min=1.0)) ** 2

        y_proj_future = project_sequence_torch(y_future, y0=float(y_train[-1]), dy_cap=dy_cap, growth_cap=growth_cap, smooth_alpha=cfg['proj_smooth_alpha_train'])
        norm_proj = torch.clamp(torch.mean(y_future.detach()) + 1.0, min=1.0)
        loss_proj_cons = torch.mean(((y_future - y_proj_future) / norm_proj) ** 2)

        rollout_steps = int(cfg.get('rollout_steps', 5))
        rollout_steps = max(2, rollout_steps)
        t_roll = np.arange(float(t_train[-1]) - rollout_steps + 1.0, float(t_train[-1]) + 1.0 + train_horizon, dtype=np.float32)
        t_roll_t = torch.tensor(t_roll, dtype=torch.float32, device=device).reshape(-1, 1)
        y_roll = torch.expm1(model.emu(t_roll_t)[:, 7])
        y_roll_proj = project_sequence_torch(y_roll, y0=float(y_train[max(0, len(y_train) - rollout_steps)]), dy_cap=dy_cap, growth_cap=growth_cap, smooth_alpha=cfg['proj_smooth_alpha_train'])

        y_roll_f = y_roll[rollout_steps:]
        y_roll_proj_f = y_roll_proj[rollout_steps:]
        dy_roll = y_roll_f[1:] - y_roll_f[:-1]
        dy_roll_proj = y_roll_proj_f[1:] - y_roll_proj_f[:-1]

        norm_roll = torch.clamp(torch.mean(y_roll_proj_f.detach()) + 1.0, min=1.0)
        loss_roll_lvl = torch.mean(((y_roll_f - y_roll_proj_f.detach()) / norm_roll) ** 2)
        loss_roll_dyn = torch.mean(((dy_roll - dy_roll_proj.detach()) / torch.clamp(dy_cap_t, min=1.0)) ** 2)
        loss_rollout = loss_roll_lvl + cfg['lambda_roll_dyn'] * loss_roll_dyn

        phys_term = (
            lambda_phys * loss_phys
            + cfg['lambda_mono'] * loss_mono
            + cfg['lambda_spike'] * loss_spike
            + cfg['lambda_growth'] * loss_growth
            + cfg['lambda_smooth'] * loss_smooth
            + cfg['lambda_anchor'] * loss_anchor
        )

        loss = loss_data + phys_term + lambda_proj * loss_proj_cons + lambda_roll * loss_rollout
        loss.backward()
        optimizer.step()
        scheduler.step(loss.detach())

        if epoch % cfg['eval_every'] == 0:
            y_val_raw = model.predict_itot_numpy(t_val)
            y_val_eval = project_sequence(y_val_raw, y0=float(y_train[-1]), dy_cap=dy_cap, growth_cap=growth_cap, smooth_alpha=cfg['proj_smooth_alpha'])
            val_rmse = float(np.sqrt(np.mean((y_val - y_val_eval) ** 2)))

            y_fore_raw = model.predict_itot_numpy(t_future_eval_np)
            y_fore_eval = project_sequence(y_fore_raw, y0=float(y_train[-1]), dy_cap=dy_cap, growth_cap=growth_cap, smooth_alpha=cfg['proj_smooth_alpha'])
            if len(y_fore_eval) > 1:
                dy_fore = np.diff(y_fore_eval)
                max_fore_jump = float(np.max(dy_fore))
                max_fore_growth = float(np.max(dy_fore / (y_fore_eval[:-1] + 1.0)))
            else:
                max_fore_jump, max_fore_growth = 0.0, 0.0

            spike_pen = max(0.0, max_fore_jump / (dy_cap + 1e-6) - 1.0)
            growth_pen = max(0.0, max_fore_growth / (growth_cap + 1e-8) - 1.0)

            score = val_rmse * (1.0 + cfg['best_spike_alpha'] * spike_pen + cfg['best_growth_alpha'] * growth_pen)

            if epoch >= cfg['best_select_start_epoch']:
                if score < best_score:
                    best_score = score
                    best_state = copy.deepcopy(model.state_dict())
                    best_epoch = epoch
                    no_improve = 0
                else:
                    no_improve += cfg['eval_every']

            if epoch >= cfg['min_epochs'] and no_improve >= cfg['early_stop_patience']:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - start

    y_train_pred = model.predict_itot_numpy(t_train)
    y_val_raw = model.predict_itot_numpy(t_val)
    y_val_pred = project_sequence(y_val_raw, y0=float(y_train[-1]), dy_cap=dy_cap, growth_cap=growth_cap, smooth_alpha=cfg['proj_smooth_alpha'])

    y_fore_raw = model.predict_itot_numpy(t_future_eval_np)
    y_fore_pred = project_sequence(y_fore_raw, y0=float(y_train[-1]), dy_cap=dy_cap, growth_cap=growth_cap, smooth_alpha=cfg['proj_smooth_alpha'])

    if len(y_fore_pred) > 1:
        dy_fore_final = np.diff(y_fore_pred)
        max_fore_jump = float(np.max(dy_fore_final))
        max_fore_growth = float(np.max(dy_fore_final / (y_fore_pred[:-1] + 1.0)))
    else:
        max_fore_jump, max_fore_growth = 0.0, 0.0

    val_mae = float(np.mean(np.abs(y_val - y_val_pred)))
    val_rmse = float(np.sqrt(np.mean((y_val - y_val_pred) ** 2)))
    val_rmse_raw = float(np.sqrt(np.mean((y_val - y_val_raw) ** 2)))

    res_chk = model.netf_data()
    residuals = [float(res_chk[i].abs().mean().detach().cpu().item()) for i in range(8)]

    row = {
        'region': region,
        'train_n': len(y_train),
        'val_n': len(y_val),
        'train_rel_rmse': rel_rmse(y_train, y_train_pred),
        'val_mae_81_90': val_mae,
        'val_rmse_81_90': val_rmse,
        'val_rmse_raw_81_90': val_rmse_raw,
        'val_nmae_pct': 100.0 * val_mae / (float(np.mean(y_val)) + 1e-9),
        'val_nrmse_pct': 100.0 * val_rmse / (float(np.mean(y_val)) + 1e-9),
        'val_nrmse_raw_pct': 100.0 * val_rmse_raw / (float(np.mean(y_val)) + 1e-9),
        'projection_dependence_index': (val_rmse_raw - val_rmse) / (val_rmse_raw + 1e-9),
        'best_epoch': best_epoch,
        'train_time_sec': elapsed,
        'max_forecast_jump': max_fore_jump,
        'max_forecast_growth': max_fore_growth,
        'dy_cap': dy_cap,
        'growth_cap': growth_cap,
        'res_f1': residuals[0],
        'res_f2': residuals[1],
        'res_f3': residuals[2],
        'res_f4': residuals[3],
        'res_f5': residuals[4],
        'res_f6': residuals[5],
        'res_f7': residuals[6],
        'res_f8': residuals[7],
    }

    print(f"[{region}] MAE={val_mae:.2f} RMSE={val_rmse:.2f} rawRMSE={val_rmse_raw:.2f} nRMSE%={row['val_nrmse_pct']:.3f} best_ep={best_epoch} time={elapsed:.1f}s")
    return row


In [ ]:
cfg_one = {
    'use_features': True,
    'lr': 4e-4,
    'epochs': 2500,
    'warmup': 600,
    'ramp': 900,
    'lambda_phys_max': 0.10,
    'lambda_mono': 0.05,
    'lambda_spike': 0.08,
    'lambda_growth': 0.14,
    'lambda_smooth': 0.05,
    'lambda_anchor': 0.15,
    'lambda_proj': 0.30,
    'proj_start_epoch': 200,
    'proj_ramp_epochs': 700,
    'lambda_roll': 0.18,
    'rollout_start_epoch': 700,
    'rollout_ramp_epochs': 800,
    'lambda_roll_dyn': 0.5,
    'rollout_steps': 5,
    'proj_horizon_min': 6,
    'proj_horizon_max': 12,
    'eval_horizon': 10,
    'time_weight_start': 0.30,
    'huber_beta': 0.45,
    'lambda_data_rel': 0.08,
    'lambda_trend': 0.45,
    'trend_tail_k': 16,
    'trend_beta': 1200.0,
    'cap_tail_window': 20,
    'spike_cap_q_tail': 0.95,
    'spike_cap_mult_tail': 1.8,
    'growth_cap_q_tail': 0.95,
    'growth_cap_mult_tail': 2.0,
    'spike_cap_q_global': 0.80,
    'spike_cap_mult_global': 0.70,
    'growth_cap_q_global': 0.80,
    'growth_cap_mult_global': 0.40,
    'anchor_tail_window': 12,
    'best_spike_alpha': 0.08,
    'best_growth_alpha': 0.08,
    'best_select_start_epoch': 600,
    'proj_smooth_alpha': 0.15,
    'proj_smooth_alpha_train': 0.10,
    'n_colloc': 160,
    'phys_ema_beta': 0.95,
    'phys_eps': 1e-8,
    'phys_weight_clip': 10.0,
    'phys_temp': 0.80,
    'eval_every': 100,
    'early_stop_patience': 700,
    'min_epochs': 1200,
}

regions = ['bello', 'cam', 'iquitos', 'phi', 'sanjuan', 'vn']
rows = []
for region in regions:
    if not (os.path.exists(f'data/{region}_train.csv') and os.path.exists(f'data/{region}_val.csv')):
        print('skip missing:', region)
        continue
    rows.append(run_experiment(region, cfg_one))

res_df = pd.DataFrame(rows)
res_df = res_df.sort_values('val_nrmse_pct').reset_index(drop=True)

os.makedirs('outputs', exist_ok=True)
out_path = 'outputs/one_config_multidata_eval.csv'
res_df.to_csv(out_path, index=False)
print('saved:', out_path)
res_df


In [ ]:
summary = {
    'regions': int(len(res_df)),
    'mean_val_nrmse_pct': float(res_df['val_nrmse_pct'].mean()),
    'median_val_nrmse_pct': float(res_df['val_nrmse_pct'].median()),
    'worst_region': str(res_df.iloc[res_df['val_nrmse_pct'].idxmax()]['region']),
    'best_region': str(res_df.iloc[res_df['val_nrmse_pct'].idxmin()]['region']),
    'mean_projection_dependence': float(res_df['projection_dependence_index'].mean()),
}
summary_df = pd.DataFrame([summary])
summary_out = 'outputs/one_config_multidata_summary.csv'
summary_df.to_csv(summary_out, index=False)
print(summary_df.to_string(index=False))
print('saved:', summary_out)
